# Fresh hybrid curriculum · four rounds on Colab

Each of four rounds uses 200 fresh examples: 60 programmatic cases, 60 teacher general cases, and 80 student-screened cases. The run ranks checkpoints by development macro score for research and retains the existing safety checks and rollback for adapter selection. Production release evaluation remains disabled.

Select a GPU, authorize the experiment, upload the matching bundle and set your cumulative API cap. The run writes to a separate Drive folder and reads prior training plans for exclusion and provenance. It evaluates the baseline once and scores all four candidates on the frozen 360-case development set. The cap covers all four rounds. If you enable macro continuation before launch, the highest development macro checkpoint may seed a later research round, while accepted checkpoint selection stays safety gated.


In [ ]:
from pathlib import Path
import os
import sys

if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("Restart the Colab session before running this notebook from the top.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/hf-cache"
for name in ("LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"):
    os.environ.pop(name, None)
# This experiment saves to Drive, independently of the ongoing Kaggle run.
os.environ.pop("KAGGLE_ADAPTER_DATASET", None)
os.environ["GITHUB_ADAPTER_UPLOAD"] = "0"
os.environ["DISTILLATION_DRIVE_BACKUP"] = "1"
os.environ["DISTILLATION_LOCAL_LOCK_DIR"] = "/content/distillation-colab-locks"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/HealthInsuranceSLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
HISTORICAL_RUN = DRIVE_ROOT / "colab_l4_balanced_5x200_v3"
CONTINUATION_AUTHORIZED = False
if CONTINUATION_AUTHORIZED is not True:
    raise RuntimeError("Explicitly authorize this four-round experiment before continuing.")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive did not mount; do not run training on temporary storage.")
print("Persistent Google Drive folder:", DRIVE_ROOT)


## Upload the companion code bundle

The ZIP contains the current loop code and fixed evaluation corpus, not model weights or credentials. This avoids a stale GitHub checkout. Upload the ZIP supplied with this notebook; it is checked against the expected SHA256 before extraction. Extraction writes only into this experiment's `/content/Health-Insurance-SLM-Colab` code folder, not Google Drive or your Kaggle run.


In [ ]:
import hashlib
import zipfile

from io import BytesIO
from tempfile import TemporaryDirectory, mkdtemp


def verified_project(project, bundle, checksum):
    payload = bundle.read_bytes() if bundle.is_file() else b""
    if hashlib.sha256(payload).hexdigest() != checksum:
        raise ValueError("Training ZIP checksum does not match")
    with zipfile.ZipFile(BytesIO(payload)) as archive:
        expected = {}
        for member in archive.infolist():
            path = Path(member.filename)
            if (path.is_absolute() or ".." in path.parts or len(path.parts) < 2
                    or path.parts[0] != project.name or member.is_dir()):
                raise ValueError("Unexpected training bundle member")
            relative = Path(*path.parts[1:])
            if relative in expected:
                raise ValueError("Duplicate training bundle member")
            expected[relative] = hashlib.sha256(archive.read(member)).hexdigest()

        def matches(directory):
            if not directory.is_dir() or directory.is_symlink():
                return False
            paths = list(directory.rglob("*"))
            if any(path.is_symlink() for path in paths):
                return False
            files = {path.relative_to(directory) for path in paths if path.is_file()}
            return files == expected.keys() and all(
                hashlib.sha256((directory / file).read_bytes()).hexdigest() == digest
                for file, digest in expected.items())

        if matches(project):
            return project
        with TemporaryDirectory(dir=project.parent) as folder:
            fresh = Path(folder) / project.name
            archive.extractall(folder)
            if not matches(fresh):
                raise ValueError("Extracted training code does not match the verified bundle")
            previous = None
            if project.exists() or project.is_symlink():
                previous = Path(mkdtemp(prefix=".stale-training-", dir=project.parent)) / project.name
                project.replace(previous)
            try:
                fresh.replace(project)
            except OSError:
                if previous is not None:
                    previous.replace(project)
                raise
        if not matches(project):
            raise ValueError("Installed training code failed verification")
        return project



EXPECTED_BUNDLE_SHA256 = "0877db1defe5fc9eae8fec7cd37d2e0b434560dbeb6a10189d32d37f51f51d4b"
BUNDLE = Path("/content/colab_distillation_bundle.zip")
PROJECT_ROOT = Path("/content/Health-Insurance-SLM-Colab")

if not BUNDLE.is_file() or hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
    from google.colab import files
    print("Upload colab_distillation_bundle.zip supplied with this notebook.")
    uploaded = files.upload()
    payload = next((data for data in uploaded.values()
                    if hashlib.sha256(data).hexdigest() == EXPECTED_BUNDLE_SHA256), None)
    if payload is None:
        raise RuntimeError("None of the uploaded files matches this notebook’s bundle checksum.")
    BUNDLE.write_bytes(payload)
    del uploaded, payload
verified_project(PROJECT_ROOT, BUNDLE, EXPECTED_BUNDLE_SHA256)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-colab.txt")
print("Code and fixed evaluation data ready:", PROJECT_ROOT)


In [ ]:
from importlib.metadata import version
import subprocess

# Preserve the CUDA-enabled PyTorch build supplied by Colab.
torch_constraint = Path("/content/distillation-torch-constraint.txt")
torch_constraint.write_text(f"torch=={version('torch')}\n")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-c", str(torch_constraint),
     "-r", REQUIREMENTS],
    check=True,
)


If Colab asks for a runtime restart after installation, restart it and rerun from the first setup cell. Your Drive run folder is preserved. The uploaded ZIP may need uploading again if the VM was replaced.


## Configure the run

A fresh run initializes a rank-32 rsLoRA adapter on Qwen3-8B. An existing run resumes from its Drive folder. This profile initializes its own adapter.


In [ ]:
import json
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import Store, atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.colab import distillation_1000_configs, save_colab_subset
from distillation_loop.preflight import readiness

RUN_NAME = "colab_l4_hybrid_fresh_4round_v1"
RESEARCH_MACRO_CONTINUATION = False  # Opt in before the first launch if desired.
TRAIN_BATCH_SIZE = 4
INFERENCE_BATCH_SIZE = 8
MODEL_NAME = "Qwen/Qwen3-8B"
RUN_DIR = DRIVE_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
parent_config, loop_config = distillation_1000_configs(
    load_config(PROJECT_ROOT / "config/self_instruct.json"),
    load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json")),
    RUN_DIR, historical_run=HISTORICAL_RUN,
)
experiment = json.loads((PROJECT_ROOT / "config/hybrid_colab_4round_v1.json").read_text())
loop_config.update(max_rounds=experiment["max_rounds"], scenarios_per_round=experiment["scenarios_per_round"],
                   research_macro_continuation=RESEARCH_MACRO_CONTINUATION,
                   hybrid_curriculum=experiment["hybrid_curriculum"], require_semantic_decision=False,
                   hybrid_prior_training_plans=[str(DRIVE_ROOT / "colab_l4_semantic_remediation_round1_v1"
                                                   / "round_01" / "training_plan.json")])
prior_hybrid = DRIVE_ROOT / "colab_l4_hybrid_fresh_round1_v2" / "round_01" / "training_plan.json"
existing_config = RUN_DIR / "colab_config.json"
if existing_config.exists():
    saved_plans = json.loads(existing_config.read_text())["hybrid_prior_training_plans"]
    if (saved_plans != loop_config["hybrid_prior_training_plans"]
            and saved_plans != [*loop_config["hybrid_prior_training_plans"], str(prior_hybrid)]):
        raise RuntimeError("Saved prior training-plan paths differ from this experiment.")
    loop_config["hybrid_prior_training_plans"] = saved_plans
elif prior_hybrid.is_file():
    loop_config["hybrid_prior_training_plans"].append(str(prior_hybrid))
parent_config["sampling"]["machine_demonstration_fraction"] = 0.0
loop_config["quality"]["minimum_new_pair_fraction"] = 1.0
loop_config["training"]["replay_fraction"] = 0.0
from distillation_loop.hybrid import options as hybrid_options
hybrid_options(loop_config)
for plan in [loop_config["historical_training_plan"], *loop_config["hybrid_prior_training_plans"]]:
    if not Path(plan).is_file():
        raise FileNotFoundError(f"Missing prior training plan required for exclusion: {plan}")
if (loop_config["max_rounds"] != 4 or loop_config["preference_training"]
        or loop_config["require_semantic_decision"] is not False
        or loop_config["stop_when_thresholds_met"]
        or not loop_config["rollback_on_regression"]):
    raise RuntimeError("Four-round experiment safety configuration mismatch.")
split = json.loads(Path(loop_config["evaluation"]["split_path"]).read_text())
if split["development"]["sha256"] != "9dd6625a3b2552a8bb780e5efa9c927fcd7caff3e55c1da741004ecc5f5664e2":
    raise RuntimeError("Unexpected frozen development split.")

# Use the balanced experiment's training and inference batch settings.
loop_config["training"].update(per_device_train_batch_size=2, gradient_accumulation_steps=2)
loop_config["inference"]["batch_size"] = 4
# Validate the profile without overwriting a previous run's configuration.
profile_path = Path("/content/requested_colab_config.json")
atomic_json(profile_path, loop_config)
loop_config = load_loop_config(str(profile_path))
report = readiness(parent_config, loop_config, check_runtime=False)
print("Readiness:", report)
if not report["ready"]:
    raise RuntimeError("Resolve the data/configuration issues above before loading the model.")
schemas = json.loads((PROJECT_ROOT / "data/self_instruct/task_schemas.json").read_text())["tasks"]
subset = save_colab_subset(loop_config, schemas, RUN_DIR)
print("Fixed evaluation cases:", subset["total"], "Per task:", subset["per_task"])

saved_config = RUN_DIR / "colab_config.json"
if saved_config.exists():
    saved_profile = json.loads(saved_config.read_text())
    # Older notebooks allowed these execution settings to be chosen before the run.
    for section, keys in (("training", ("per_device_train_batch_size", "gradient_accumulation_steps")),
                          ("inference", ("batch_size",))):
        for key in keys:
            loop_config[section][key] = saved_profile[section][key]
    if saved_profile != loop_config:
        raise RuntimeError("This Drive run folder has different settings; keep the original settings or choose a new RUN_NAME.")
# Confirm Drive supports the actual journal write and local process-lock path.
probe = Store(RUN_DIR / "storage_probe")
with probe.lock():
    probe.append("probe.jsonl", {"drive_write_test": True})
(probe.directory / "probe.jsonl").unlink()
atomic_json(saved_config, loop_config)
print("Google Drive writes verified:", RUN_DIR)
print(f'{loop_config["max_rounds"]} rounds × {loop_config["scenarios_per_round"]} new pairs')
print(f'{loop_config["adaptive_pairs_per_round"]} adaptive pairs per round | '
      f'requested training batch {TRAIN_BATCH_SIZE} | inference batch {INFERENCE_BATCH_SIZE}')


In [ ]:
import getpass

if not os.environ.get("OPENAI_API_KEY", "").strip():
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (hidden): ").strip()
if not os.environ["OPENAI_API_KEY"]:
    raise RuntimeError("OPENAI_API_KEY is required for teacher training-data generation.")
print("Teacher credential loaded. It will not be written to Google Drive or adapter archives.")


## Load the student model

This cell loads Qwen3-8B and reports the assigned GPU. The run cell uses the teacher API to grade open-ended benchmark answers and generate training pairs.


In [ ]:
# Import Unsloth before Transformers, PEFT or TRL so its patches apply.
if CONTINUATION_AUTHORIZED is not True:
    raise RuntimeError("Explicitly authorize this four-round experiment before loading the model.")
import unsloth
import torch
from distillation_loop.colab import UnslothStudentRuntime, l4_runtime_config
from distillation_loop.evaluation import require_training_versions

require_training_versions()
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
runtime_config = l4_runtime_config(
    loop_config, train_batch_size=TRAIN_BATCH_SIZE, inference_batch_size=INFERENCE_BATCH_SIZE,
)
student_runtime = UnslothStudentRuntime.load(MODEL_NAME, runtime_config)
print("Unsloth adapter ready. Inference batch size:", INFERENCE_BATCH_SIZE)
torch.cuda.reset_peak_memory_stats()


## Start or resume training

Run this cell when you’re ready to begin. The teacher API budget is cumulative across sessions. On resume, press Enter to reuse the saved cap or enter a higher cap. Keep the same Drive folder and settings so the notebook can reuse saved progress.



In [ ]:
if CONTINUATION_AUTHORIZED is not True:
    raise RuntimeError("Explicitly authorize this four-round experiment before starting.")

from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget, BudgetExceeded
from self_instruct_pipeline.usage import export_usage

from google.colab import userdata
from self_instruct_pipeline.usage_cloud import SupabaseLedger, sync_usage_history

for name in ("SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    if not os.environ.get(name):
        try:
            os.environ[name] = userdata.get(name).strip()
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
if SupabaseLedger.from_environment() is None:
    raise RuntimeError("Enable SUPABASE_URL and SUPABASE_SECRET_KEY in Colab Secrets.")
print("Supabase usage sync enabled.", sync_usage_history(RUN_DIR))

budget_path = RUN_DIR / "api_budget" / "state.json"
saved_limits = json.loads(budget_path.read_text())["limits"] if budget_path.exists() else None
prompt = "Cumulative teacher and judge API cap in USD"
if saved_limits:
    prompt += f' [Enter to keep ${saved_limits["max_cost_usd"]}]'
budget_usd = input(prompt + ": ").strip()
if not budget_usd and saved_limits:
    budget_usd = saved_limits["max_cost_usd"]
if not budget_usd:
    raise ValueError("Enter a teacher API budget before starting.")
max_calls = max(10000, saved_limits["max_calls"]) if saved_limits else 10000
budget = APIBudget(RUN_DIR / "api_budget", max_calls=max_calls, max_cost_usd=budget_usd)

try:
    with budget.activate(allow_increase=True):
        result = run_loop(student_runtime, parent_config, loop_config, evaluate_release=False)
except BudgetExceeded as exc:
    print("Paused at teacher API budget:", exc)
    print("Rerun this cell with a higher cumulative cap to resume.")
else:
    print("Stop reason:", result["status"])
    print("Macro ranking and safety:", result.get("experimental_macro_rank", []))
    print("Selected checkpoint:", result["best_checkpoint"])
finally:
    print("Peak GPU memory reserved (GiB):", round(torch.cuda.max_memory_reserved() / 1024**3, 1))
    print("Google Drive run folder:", RUN_DIR)
    print("Usage:", export_usage(RUN_DIR))


In [ ]:
# Inspect persistent backups and progress without training or API calls.
for receipt_path in sorted((RUN_DIR / "drive_backups").glob("round_*.json")):
    receipt = json.loads(receipt_path.read_text())
    print(receipt["status"], receipt["archive"])
state_path = RUN_DIR / "loop_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    print("Completed rounds:", state["completed_rounds"], "Status:", state["status"])
    for entry in state["history"]:
        print("Round", entry["round"], "macro", entry["scores"]["macro_task_score"])


In [ ]:
# Retry only the latest Drive ZIP export if a backup operation was interrupted.
# This does not retrain the model or regenerate teacher data.
from distillation_loop.drive_backup import save_drive_adapter
completed = sorted(RUN_DIR.glob("round_*/training_complete.json"),
                   key=lambda path: int(path.parent.name.split("_")[-1]))
if completed:
    latest = completed[-1]
    checkpoint = json.loads(latest.read_text())["checkpoint"]
    save_drive_adapter(checkpoint, RUN_DIR, int(latest.parent.name.split("_")[-1]), require_mounted=True)
else:
    print("No completed round yet. Initial adapter/state may still be available on Drive.")
